# One-Year ETF Price Comparison

Compare daily adjusted closing prices from `daily_prices` for five representative ETFs in the asset universe:

- `SPY`: S&P 500
- `QQQ`: Nasdaq-100
- `VTI`: total U.S. stock market
- `VXUS`: international equities
- `AGG`: U.S. aggregate bonds

The query joins `daily_prices` to `assets`, filters to the trailing 365 days, and uses `close` when `adjusted_close` is missing.

In [ ]:
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import bindparam, create_engine, text

from trade_tools.config import config


tickers = ["SPY", "QQQ", "VTI", "VXUS", "AGG"]
start_date = datetime.now(timezone.utc) - timedelta(days=365)

query = text(
    """
    SELECT a.ticker,
           dp.timestamp,
           COALESCE(dp.adjusted_close, dp.close) AS price
    FROM daily_prices AS dp
    JOIN assets AS a ON a.id = dp.asset_id
    WHERE a.type = 'etf'
      AND a.active = TRUE
      AND a.ticker IN :tickers
      AND dp.timestamp >= :start_date
      AND COALESCE(dp.adjusted_close, dp.close) IS NOT NULL
    ORDER BY dp.timestamp
    """
).bindparams(bindparam("tickers", expanding=True))

engine = create_engine(config.DATABASE_URL, pool_pre_ping=True)
try:
    with engine.connect() as connection:
        prices = pd.read_sql_query(
            query,
            connection,
            params={"tickers": tickers, "start_date": start_date},
        )
finally:
    engine.dispose()

if prices.empty:
    raise ValueError("No daily ETF prices found in the database for the past year.")

prices["timestamp"] = pd.to_datetime(prices["timestamp"], utc=True)
price_data = (
    prices.pivot(index="timestamp", columns="ticker", values="price")
    .sort_index()
    .reindex(columns=[ticker for ticker in tickers if ticker in prices["ticker"].unique()])
)

missing_tickers = sorted(set(tickers) - set(price_data.columns))
if missing_tickers:
    print(f"No price data found for: {', '.join(missing_tickers)}")

ax = price_data.plot(figsize=(12, 6), linewidth=1.5)
ax.set_title("Daily Adjusted Close Prices: Five Core ETFs")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.grid(True, alpha=0.3)
ax.legend(title="Ticker")
plt.tight_layout()
plt.show()